In [6]:
# ----- 2. 월별 패널 데이터 생성 함수 -----
def create_monthly_panel(dataframe, new_emotion_cols):
    """월별 패널 데이터를 생성하는 함수"""
    # 월별 집계
    dataframe['month_start'] = dataframe['DATE'].dt.to_period('M').apply(lambda p: p.start_time)
    
    # 집계 규칙: 감정 컬럼들은 평균, 나머지는 첫번째 값
    agg_dict = {col: 'mean' for col in new_emotion_cols}
    for col in ['OTT공개일', '대표국적', '장르']:
        if col in dataframe.columns:
            agg_dict[col] = 'first'
    
    df_monthly_agg = dataframe.groupby(['영화명', 'month_start']).agg(agg_dict).reset_index()

    # 균형 패널 생성
    date_range = pd.date_range(start='2023-01-01', end='2024-12-31', freq='MS')
    movie_list = df_monthly_agg['영화명'].unique()
    panel_skeleton = pd.MultiIndex.from_product([movie_list, date_range], names=['영화명', 'month_start'])
    panel_df = pd.DataFrame(index=panel_skeleton).reset_index()
    df_panel = pd.merge(panel_df, df_monthly_agg, on=['영화명', 'month_start'], how='left')

    # 메타데이터 채우기
    cols_to_fill = ['OTT공개일', '대표국적', '장르']
    for col in cols_to_fill:
        if col in df_panel.columns:
            df_panel[col] = df_panel.groupby('영화명')[col].ffill().bfill()

    # CSDID용 변수 생성
    first_month = df_panel['month_start'].min()
    df_panel['month_index'] = ((df_panel['month_start'].dt.year - first_month.year) * 12 +
                               (df_panel['month_start'].dt.month - first_month.month))
    df_panel['gvar_month_start'] = df_panel['OTT공개일'].dt.to_period('M').apply(lambda p: p.start_time)
    df_panel['gvar_month'] = ((df_panel['gvar_month_start'].dt.year - first_month.year) * 12 +
                              (df_panel['gvar_month_start'].dt.month - first_month.month))
    
    return df_panel


# ----- 3. 방법 1: 평균(mean) 방식 -----
print("\n2. 평균(mean) 방식으로 새로운 감정 확률을 계산합니다...")
df_mean = df_filtered.copy()
mean_cols = []


2. 평균(mean) 방식으로 새로운 감정 확률을 계산합니다...


In [8]:
# 각 8개 감정에 대한 가중 평균 계산
for emotion in PRIMARY_EMOTIONS:
    new_col_name = f"P_{emotion}"
    mean_cols.append(new_col_name)
    
    # 가중치를 적용할 컬럼과 가중치를 저장할 리스트
    weighted_cols = []
    weights = []
    
    # 1. 직접 매핑된 컬럼 추가 (가중치 1)
    direct_cols = globals()[f"{emotion}_COLS"]
    for col in direct_cols:
        if col in df_mean.columns:
            weighted_cols.append(df_mean[col])
            weights.append(1)
            
    # 2. 혼합 매핑된 컬럼 추가 (가중치 0.5)
    for blend_col, mapping in BLEND_MAP.items():
        if emotion in mapping and blend_col in df_mean.columns:
            weighted_cols.append(df_mean[blend_col])
            weights.append(mapping[emotion])
            
    # 가중 평균 계산: (값 * 가중치)의 합 / 가중치의 합
    if weighted_cols:
        weighted_sum = np.sum([col * w for col, w in zip(weighted_cols, weights)], axis=0)
        sum_of_weights = np.sum(weights)
        df_mean[new_col_name] = weighted_sum / sum_of_weights
    else:
        df_mean[new_col_name] = 0


In [10]:
# 월별 패널로 만들고 저장
panel_mean = create_monthly_panel(df_mean, mean_cols)
panel_mean.to_stata('movie_monthly_panel_mean.dta', write_index=False, version=118)
print("'movie_monthly_panel_mean.dta' 파일 저장 완료.")


# ----- 4. 방법 2: 최댓값(max) 방식 -----
print("\n3. 최댓값(max) 방식으로 새로운 감정 확률을 계산합니다...")
df_max = df_filtered.copy()
max_cols = []

# 각 8개 감정에 대한 최댓값 계산
for emotion in PRIMARY_EMOTIONS:
    new_col_name = f"M_{emotion}"
    max_cols.append(new_col_name)
    
    candidate_cols = []
    
    # 1. 직접 매핑된 컬럼
    candidate_cols.extend(globals()[f"{emotion}_COLS"])
    
    # 2. 혼합 매핑된 컬럼
    for blend_col, mapping in BLEND_MAP.items():
        if emotion in mapping:
            candidate_cols.append(blend_col)
            
    # DataFrame에 존재하는 컬럼만 필터링
    existing_candidates = [col for col in candidate_cols if col in df_max.columns]
    
    if existing_candidates:
        df_max[new_col_name] = df_max[existing_candidates].max(axis=1)
    else:
        df_max[new_col_name] = 0


'movie_monthly_panel_mean.dta' 파일 저장 완료.

3. 최댓값(max) 방식으로 새로운 감정 확률을 계산합니다...


In [12]:
# 월별 패널로 만들고 저장
panel_max = create_monthly_panel(df_max, max_cols)
panel_max.to_stata('movie_monthly_panel_max.dta', write_index=False, version=118)
print("'movie_monthly_panel_max.dta' 파일 저장 완료.")

print("\n✨ 모든 작업 완료! ✨")

'movie_monthly_panel_max.dta' 파일 저장 완료.

✨ 모든 작업 완료! ✨


# 2020년부터 2025년까지 전체

In [6]:
# -*- coding: utf-8 -*-
"""
Make monthly panel (MEAN / MAX) for 8 primary emotions from the raw micro CSV,
keeping never-treated titles and using the full calendar range in the data.

Outputs:
  - movie_monthly_panel_mean.dta
  - movie_monthly_panel_max.dta
"""

import pandas as pd
import numpy as np

# =========================
# 0) SETTINGS / PATH
# =========================
INPUT_CSV = "movie_long_with_emotions_full.csv"   # 원본 CSV 경로
STATA_VERSION = 118                               # .dta 버전 (Stata 17 호환)
META_COLS = ["OTT공개일", "대표국적", "장르"]        # 있으면 패널에 유지/전파

# (선택) 달력 범위를 고정하고 싶으면 아래 2개를 문자열로 지정 (예: "2020-01-01", "2024-12-01")
FIXED_RANGE_START = None  # e.g., "2020-01-01"
FIXED_RANGE_END   = None  # e.g., "2024-12-01"

# =========================
# 1) Emotion column mapping
# =========================
# (중복 제거 + 존재하는 컬럼만 사용 예정)
JOY_COLS = list(set([
    "기쁨_prob","행복_prob","즐거움/신남_prob","흐뭇함(귀여움/예쁨)_prob",
    "흐뭇함_귀여움_예쁨__prob","뿌듯함_prob","감동/감탄_prob","편안/쾌적_prob"
]))
TRUST_COLS = list(set(["안심/신뢰_prob","환영/호의_prob","존경_prob","아껴주는_prob"]))
FEAR_COLS  = list(set(["공포/무서움_prob","불안/걱정_prob","부담/안_내킴_prob"]))
SURPRISE_COLS = list(set(["놀람_prob","경악_prob","당황/난처_prob","깨달음_prob"]))
SADNESS_COLS  = list(set([
    "슬픔_prob","절망_prob","서러움_prob","안타까움/실망_prob","재미없음_prob",
    "힘듦/지침_prob","패배/자기혐오_prob","죄책감_prob","불쌍함/연민_prob"
]))
DISGUST_COLS = list(set(["역겨움/징그러움_prob","한심함_prob","어이없음_prob","지긋지긋_prob"]))
ANGER_COLS   = list(set(["화남/분노_prob","짜증_prob","불평/불만_prob"]))
ANTICIP_COLS = list(set(["기대감_prob","신기함/관심_prob","의심/불신_prob"]))

# 한 컬럼을 두 감정에 분배(혼합 매핑)
BLEND_MAP = {
    "증오/혐오_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "우쭐댐/무시함_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "고마움_prob": {"JOY": 0.5, "TRUST": 0.5},
    "부끄러움_prob": {"SADNESS": 0.5, "FEAR": 0.5},
}

PRIMARY_EMOTIONS = ["JOY", "TRUST", "FEAR", "SURPRISE", "SADNESS", "DISGUST", "ANGER", "ANTICIP"]
EMOTION_TO_COLS = {
    "JOY": JOY_COLS, "TRUST": TRUST_COLS, "FEAR": FEAR_COLS, "SURPRISE": SURPRISE_COLS,
    "SADNESS": SADNESS_COLS, "DISGUST": DISGUST_COLS, "ANGER": ANGER_COLS, "ANTICIP": ANTICIP_COLS
}

# =========================
# 2) Utilities
# =========================
def read_csv_kor(path: str) -> pd.DataFrame:
    """한국어 CSV 인코딩에 안전하게 로드"""
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(path)

def to_float_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype(float)

def compute_mean_emotions(df: pd.DataFrame) -> pd.DataFrame:
    """가중 평균(혼합 매핑 포함)으로 P_{EMOTION} 열 생성"""
    out = df.copy()
    for emo in PRIMARY_EMOTIONS:
        new_col = f"P_{emo}"
        weighted_cols = []
        weights = []

        # 직접 매핑(가중치 1)
        for col in EMOTION_TO_COLS[emo]:
            if col in out.columns:
                weighted_cols.append(to_float_series(out[col]))
                weights.append(1.0)

        # 혼합 매핑(예: 증오/혐오 → DISGUST/ANGER 0.5씩)
        for blend_col, mapping in BLEND_MAP.items():
            if emo in mapping and blend_col in out.columns:
                weighted_cols.append(to_float_series(out[blend_col]))
                weights.append(float(mapping[emo]))

        if weighted_cols:
            wsum = np.zeros(len(out), dtype=float)
            for series, w in zip(weighted_cols, weights):
                # NaN은 0으로 더하고, 나중에 가중치로 평균
                wsum += series.fillna(0).to_numpy() * w
            wtot = float(np.sum(weights))
            out[new_col] = wsum / wtot
        else:
            out[new_col] = np.nan
    return out

def compute_max_emotions(df: pd.DataFrame) -> pd.DataFrame:
    """최댓값으로 M_{EMOTION} 열 생성 (혼합 컬럼도 후보에 포함)"""
    out = df.copy()
    for emo in PRIMARY_EMOTIONS:
        new_col = f"M_{emo}"
        candidates = list(EMOTION_TO_COLS[emo]) + list(BLEND_MAP.keys())
        exist = [c for c in candidates if c in out.columns]
        if exist:
            out[new_col] = out[exist].apply(lambda r: to_float_series(r).max(), axis=1)
        else:
            out[new_col] = np.nan
    return out

def create_monthly_panel(dataframe: pd.DataFrame, emotion_cols: list) -> pd.DataFrame:
    """
    월 패널 생성:
      - month_start: 각 달 1일
      - movie×month 평균 (감정값) + review_count(있으면 감상평 개수, 없으면 행 수)
      - 전체 달력 범위: 데이터의 최소~최대 월 자동(또는 FIXED_RANGE_* 지정 시 고정)
      - OTT공개일은 영화 단위로 ffill/bfill
      - month_index: 패널 최초 월 기준 0,1,2,...
      - gvar_month: 공개월(월단위) 기준 index, 없으면 NaN (never-treated 유지)
    """
    df = dataframe.copy()
    df["month_start"] = df["DATE"].dt.to_period("M").dt.start_time

    has_review_text = ("감상평" in df.columns)

    # 집계: Named Aggregation으로 통일
    agg_dict = {col: (col, "mean") for col in emotion_cols}
    if has_review_text:
        agg_dict["review_count"] = ("감상평", "count")
    for c in META_COLS:
        if c in df.columns:
            agg_dict[c] = (c, "first")

    df_monthly_agg = df.groupby(["영화명", "month_start"]).agg(**agg_dict).reset_index()

    # 텍스트 열이 없으면 셀 사이즈로 review_count 생성
    if not has_review_text:
        sz = df.groupby(["영화명", "month_start"]).size().reset_index(name="review_count")
        df_monthly_agg = df_monthly_agg.merge(sz, on=["영화명", "month_start"], how="left")

    # 달력 스켈레톤: 동적 or 고정
    if FIXED_RANGE_START and FIXED_RANGE_END:
        date_range = pd.date_range(start=pd.Timestamp(FIXED_RANGE_START),
                                   end=pd.Timestamp(FIXED_RANGE_END), freq="MS")
    else:
        min_m = df["DATE"].dt.to_period("M").min().to_timestamp(how="start")
        max_m = df["DATE"].dt.to_period("M").max().to_timestamp(how="start")
        date_range = pd.date_range(start=min_m, end=max_m, freq="MS")

    movies = df_monthly_agg["영화명"].unique()
    skeleton = pd.MultiIndex.from_product([movies, date_range], names=["영화명", "month_start"])
    panel = pd.DataFrame(index=skeleton).reset_index()

    # 스켈레톤과 병합 (모든 영화×모든 달)
    panel = panel.merge(df_monthly_agg, on=["영화명", "month_start"], how="left")

    # 메타 전파
    for c in META_COLS:
        if c in panel.columns:
            panel[c] = panel.groupby("영화명")[c].ffill().bfill()

    # month_index: 패널 최초 월 기준
    first_month = panel["month_start"].min()
    panel["month_index"] = (
        (panel["month_start"].dt.year - first_month.year) * 12
        + (panel["month_start"].dt.month - first_month.month)
    )

    # gvar_month: 공개월(월단위) → index (NaT/NaN 허용: never-treated 유지)
    if "OTT공개일" in panel.columns:
        panel["OTT공개일"] = pd.to_datetime(panel["OTT공개일"], errors="coerce")
        gstart = panel["OTT공개일"].dt.to_period("M").dt.start_time
    else:
        gstart = pd.NaT

    panel["gvar_month"] = np.where(
        pd.notna(gstart),
        (gstart.dt.year - first_month.year) * 12 + (gstart.dt.month - first_month.month),
        np.nan
    )

    return panel

def coverage_summary(panel: pd.DataFrame, name: str) -> None:
    print(f"\n[{name}] rows={len(panel):,}, movies={panel['영화명'].nunique():,}, "
          f"months={panel['month_start'].nunique():,}")
    print(f"  review_count not-null cells: {panel['review_count'].notna().sum():,}")
    print(f"  treated (gvar_month notna) cells: {panel['gvar_month'].notna().sum():,}")

# =========================


In [8]:
# 3) MAIN
# =========================
def main():
    print("1) Load CSV ...")
    df = read_csv_kor(INPUT_CSV)

    print("2) Parse dates / build OTT release date ...")
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df = df[df["DATE"].notna()].copy()

    # post_netflix 기반으로 최초 OTT 공개일 산출 (있으면 활용)
    if "post_netflix" in df.columns:
        df["post_netflix"] = pd.to_numeric(df["post_netflix"], errors="coerce")
        ott_first = df[df["post_netflix"] == 1][["영화명", "DATE"]].copy()
        ott_dates = ott_first.groupby("영화명")["DATE"].min().rename("OTT공개일").reset_index()
    else:
        ott_dates = pd.DataFrame(columns=["영화명", "OTT공개일"])

    # 기존 OTT공개일과 병합(계산된 값이 있으면 우선)
    df = df.merge(ott_dates, on="영화명", how="left", suffixes=("", "_calc"))
    if "OTT공개일_calc" in df.columns:
        if "OTT공개일" in df.columns:
            df["OTT공개일"] = np.where(
                pd.notna(df["OTT공개일_calc"]), df["OTT공개일_calc"], df["OTT공개일"]
            )
        else:
            df["OTT공개일"] = df["OTT공개일_calc"]
        df.drop(columns=["OTT공개일_calc"], inplace=True)
    df["OTT공개일"] = pd.to_datetime(df["OTT공개일"], errors="coerce")  # NaT 허용(never-treated)

    # 3) MEAN 패널
    print("3) Build MEAN-based emotion probabilities ...")
    df_mean = compute_mean_emotions(df)
    mean_cols = [f"P_{e}" for e in PRIMARY_EMOTIONS]
    panel_mean = create_monthly_panel(df_mean, mean_cols)
    print(" -> Save: movie_monthly_panel_mean.dta")
    panel_mean.to_stata("movie_monthly_panel_mean.dta", write_index=False, version=STATA_VERSION)
    coverage_summary(panel_mean, "MEAN panel")

    # 4) MAX 패널
    print("4) Build MAX-based emotion probabilities ...")
    df_max = compute_max_emotions(df)
    max_cols = [f"M_{e}" for e in PRIMARY_EMOTIONS]
    panel_max = create_monthly_panel(df_max, max_cols)
    print(" -> Save: movie_monthly_panel_max.dta")
    panel_max.to_stata("movie_monthly_panel_max.dta", write_index=False, version=STATA_VERSION)
    coverage_summary(panel_max, "MAX panel")

    print("\n✨ All done.")

if __name__ == "__main__":
    main()


1) Load CSV ...
2) Parse dates / build OTT release date ...
3) Build MEAN-based emotion probabilities ...
 -> Save: movie_monthly_panel_mean.dta

[MEAN panel] rows=12,738, movies=193, months=66
  review_count not-null cells: 3,481
  treated (gvar_month notna) cells: 12,738
4) Build MAX-based emotion probabilities ...
 -> Save: movie_monthly_panel_max.dta

[MAX panel] rows=12,738, movies=193, months=66
  review_count not-null cells: 3,481
  treated (gvar_month notna) cells: 12,738

✨ All done.


# 2020년부터 2025년까지 전체(csv도 추가해서 만들기)

In [4]:
# -*- coding: utf-8 -*-
"""
Make monthly panel (MEAN / MAX) for 8 primary emotions from the raw micro CSV,
keeping never-treated titles and using the full calendar range in the data.

Outputs:
  - movie_monthly_panel_mean.dta, movie_monthly_panel_mean.csv
  - movie_monthly_panel_max.dta, movie_monthly_panel_max.csv
"""

import pandas as pd
import numpy as np

# =========================
# 0) SETTINGS / PATH
# =========================
INPUT_CSV = "movie_long_with_emotions_full.csv"      # 원본 CSV 경로
STATA_VERSION = 118                                  # .dta 버전 (Stata 17 호환)
META_COLS = ["OTT공개일", "대표국적", "장르"]           # 있으면 패널에 유지/전파

# (선택) 달력 범위를 고정하고 싶으면 아래 2개를 문자열로 지정 (예: "2020-01-01", "2024-12-01")
FIXED_RANGE_START = None  # e.g., "2020-01-01"
FIXED_RANGE_END   = None  # e.g., "2024-12-01"

# =========================
# 1) Emotion column mapping
# =========================
# (중복 제거 + 존재하는 컬럼만 사용 예정)
JOY_COLS = list(set([
    "기쁨_prob","행복_prob","즐거움/신남_prob","흐뭇함(귀여움/예쁨)_prob",
    "흐뭇함_귀여움_예쁨__prob","뿌듯함_prob","감동/감탄_prob","편안/쾌적_prob"
]))
TRUST_COLS = list(set(["안심/신뢰_prob","환영/호의_prob","존경_prob","아껴주는_prob"]))
FEAR_COLS  = list(set(["공포/무서움_prob","불안/걱정_prob","부담/안_내킴_prob"]))
SURPRISE_COLS = list(set(["놀람_prob","경악_prob","당황/난처_prob","깨달음_prob"]))
SADNESS_COLS  = list(set([
    "슬픔_prob","절망_prob","서러움_prob","안타까움/실망_prob","재미없음_prob",
    "힘듦/지침_prob","패배/자기혐오_prob","죄책감_prob","불쌍함/연민_prob"
]))
DISGUST_COLS = list(set(["역겨움/징그러움_prob","한심함_prob","어이없음_prob","지긋지긋_prob"]))
ANGER_COLS   = list(set(["화남/분노_prob","짜증_prob","불평/불만_prob"]))
ANTICIP_COLS = list(set(["기대감_prob","신기함/관심_prob","의심/불신_prob"]))

# 한 컬럼을 두 감정에 분배(혼합 매핑)
BLEND_MAP = {
    "증오/혐오_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "우쭐댐/무시함_prob": {"DISGUST": 0.5, "ANGER": 0.5},
    "고마움_prob": {"JOY": 0.5, "TRUST": 0.5},
    "부끄러움_prob": {"SADNESS": 0.5, "FEAR": 0.5},
}

PRIMARY_EMOTIONS = ["JOY", "TRUST", "FEAR", "SURPRISE", "SADNESS", "DISGUST", "ANGER", "ANTICIP"]
EMOTION_TO_COLS = {
    "JOY": JOY_COLS, "TRUST": TRUST_COLS, "FEAR": FEAR_COLS, "SURPRISE": SURPRISE_COLS,
    "SADNESS": SADNESS_COLS, "DISGUST": DISGUST_COLS, "ANGER": ANGER_COLS, "ANTICIP": ANTICIP_COLS
}

# =========================
# 2) Utilities
# =========================
def read_csv_kor(path: str) -> pd.DataFrame:
    """한국어 CSV 인코딩에 안전하게 로드"""
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(path)

def to_float_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype(float)

def compute_mean_emotions(df: pd.DataFrame) -> pd.DataFrame:
    """가중 평균(혼합 매핑 포함)으로 P_{EMOTION} 열 생성"""
    out = df.copy()
    for emo in PRIMARY_EMOTIONS:
        new_col = f"P_{emo}"
        weighted_cols = []
        weights = []

        # 직접 매핑(가중치 1)
        for col in EMOTION_TO_COLS[emo]:
            if col in out.columns:
                weighted_cols.append(to_float_series(out[col]))
                weights.append(1.0)

        # 혼합 매핑(예: 증오/혐오 → DISGUST/ANGER 0.5씩)
        for blend_col, mapping in BLEND_MAP.items():
            if emo in mapping and blend_col in out.columns:
                weighted_cols.append(to_float_series(out[blend_col]))
                weights.append(float(mapping[emo]))

        if weighted_cols:
            wsum = np.zeros(len(out), dtype=float)
            for series, w in zip(weighted_cols, weights):
                wsum += series.fillna(0).to_numpy() * w
            wtot = float(np.sum(weights))
            out[new_col] = wsum / wtot
        else:
            out[new_col] = np.nan
    return out

def compute_max_emotions(df: pd.DataFrame) -> pd.DataFrame:
    """최댓값으로 M_{EMOTION} 열 생성 (혼합 컬럼도 후보에 포함)"""
    out = df.copy()
    for emo in PRIMARY_EMOTIONS:
        new_col = f"M_{emo}"
        candidates = list(EMOTION_TO_COLS[emo]) + list(BLEND_MAP.keys())
        exist = [c for c in candidates if c in out.columns]
        if exist:
            # apply 대신 더 빠른 방식 사용
            out[new_col] = out[exist].apply(pd.to_numeric, errors='coerce').max(axis=1)
        else:
            out[new_col] = np.nan
    return out

def create_monthly_panel(dataframe: pd.DataFrame, emotion_cols: list) -> pd.DataFrame:
    """월별 패널 생성"""
    df = dataframe.copy()
    df["month_start"] = df["DATE"].dt.to_period("M").dt.start_time

    has_review_text = ("감상평" in df.columns)
    
    # 집계: Named Aggregation으로 통일
    agg_dict = {col: (col, "mean") for col in emotion_cols}
    if has_review_text:
        agg_dict["review_count"] = ("감상평", "count")
    for c in META_COLS:
        if c in df.columns:
            agg_dict[c] = (c, "first")

    df_monthly_agg = df.groupby(["영화명", "month_start"]).agg(**agg_dict).reset_index()

    # 텍스트 열이 없으면 셀 사이즈로 review_count 생성
    if not has_review_text:
        sz = df.groupby(["영화명", "month_start"]).size().reset_index(name="review_count")
        df_monthly_agg = df_monthly_agg.merge(sz, on=["영화명", "month_start"], how="left")

    # 달력 스켈레톤: 동적 or 고정
    if FIXED_RANGE_START and FIXED_RANGE_END:
        date_range = pd.date_range(start=pd.Timestamp(FIXED_RANGE_START),
                                   end=pd.Timestamp(FIXED_RANGE_END), freq="MS")
    else:
        min_m = df["DATE"].dt.to_period("M").min().to_timestamp(how="start")
        max_m = df["DATE"].dt.to_period("M").max().to_timestamp(how="start")
        date_range = pd.date_range(start=min_m, end=max_m, freq="MS")

    movies = df["영화명"].unique()
    skeleton = pd.MultiIndex.from_product([movies, date_range], names=["영화명", "month_start"])
    panel = pd.DataFrame(index=skeleton).reset_index()

    # 스켈레톤과 병합 (모든 영화×모든 달)
    panel = panel.merge(df_monthly_agg, on=["영화명", "month_start"], how="left")
    
    # 메타 전파
    for c in META_COLS:
        if c in panel.columns:
            panel[c] = panel.groupby("영화명")[c].transform('ffill').transform('bfill')

    # month_index: 패널 최초 월 기준
    first_month = panel["month_start"].min()
    panel["month_index"] = (
        (panel["month_start"].dt.year - first_month.year) * 12
        + (panel["month_start"].dt.month - first_month.month)
    )

    # gvar_month: 공개월(월단위) → index (NaT/NaN 허용: never-treated 유지)
    if "OTT공개일" in panel.columns:
        panel["OTT공개일"] = pd.to_datetime(panel["OTT공개일"], errors="coerce")
        gstart = panel["OTT공개일"].dt.to_period("M").dt.start_time
        panel["gvar_month"] = np.where(
            pd.notna(gstart),
            (gstart.dt.year - first_month.year) * 12 + (gstart.dt.month - first_month.month),
            np.nan
        )
    else:
        panel["gvar_month"] = np.nan

    return panel

def coverage_summary(panel: pd.DataFrame, name: str) -> None:
    print(f"\n[{name}] rows={len(panel):,}, movies={panel['영화명'].nunique():,}, "
          f"months={panel['month_start'].nunique():,}")
    print(f"  review_count not-null cells: {panel['review_count'].notna().sum():,}")
    print(f"  treated (gvar_month notna) cells: {panel['gvar_month'].notna().sum():,}")

In [6]:
# =========================
# 3) MAIN
# =========================
def main():
    print("1) Load CSV ...")
    df = read_csv_kor(INPUT_CSV)

    print("2) Parse dates / build OTT release date ...")
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df = df.dropna(subset=["DATE"]).copy()

    # post_netflix 기반으로 최초 OTT 공개일 산출 (있으면 활용)
    if "post_netflix" in df.columns:
        df["post_netflix"] = pd.to_numeric(df["post_netflix"], errors="coerce")
        ott_first = df.loc[df["post_netflix"] == 1, ["영화명", "DATE"]]
        ott_dates = ott_first.groupby("영화명")["DATE"].min().rename("OTT공개일").reset_index()
        
        # 기존 OTT공개일과 병합(계산된 값이 있으면 우선)
        if "OTT공개일" in df.columns:
            df = df.merge(ott_dates, on="영화명", how="left", suffixes=("", "_calc"))
            df["OTT공개일"] = df["OTT공개일_calc"].fillna(df["OTT공개일"])
            df.drop(columns=["OTT공개일_calc"], inplace=True)
        else:
            df = df.merge(ott_dates, on="영화명", how="left")
            
    if "OTT공개일" in df.columns:
        df["OTT공개일"] = pd.to_datetime(df["OTT공개일"], errors="coerce")  # NaT 허용(never-treated)

    # 3) MEAN 패널
    print("3) Build MEAN-based emotion probabilities ...")
    df_mean = compute_mean_emotions(df)
    mean_cols = [f"P_{e}" for e in PRIMARY_EMOTIONS]
    panel_mean = create_monthly_panel(df_mean, mean_cols)
    print(" -> Save: movie_monthly_panel_mean.dta / .csv")
    panel_mean.to_stata("movie_monthly_panel_mean.dta", write_index=False, version=STATA_VERSION)
    panel_mean.to_csv("movie_monthly_panel_mean.csv", index=False, encoding="utf-8-sig")
    coverage_summary(panel_mean, "MEAN panel")

    # 4) MAX 패널
    print("4) Build MAX-based emotion probabilities ...")
    df_max = compute_max_emotions(df)
    max_cols = [f"M_{e}" for e in PRIMARY_EMOTIONS]
    panel_max = create_monthly_panel(df_max, max_cols)
    print(" -> Save: movie_monthly_panel_max.dta / .csv")
    panel_max.to_stata("movie_monthly_panel_max.dta", write_index=False, version=STATA_VERSION)
    panel_max.to_csv("movie_monthly_panel_max.csv", index=False, encoding="utf-8-sig")
    coverage_summary(panel_max, "MAX panel")

    print("\n✨ All done.")

if __name__ == "__main__":
    main()

1) Load CSV ...
2) Parse dates / build OTT release date ...
3) Build MEAN-based emotion probabilities ...
 -> Save: movie_monthly_panel_mean.dta / .csv

[MEAN panel] rows=12,738, movies=193, months=66
  review_count not-null cells: 3,481
  treated (gvar_month notna) cells: 12,738
4) Build MAX-based emotion probabilities ...
 -> Save: movie_monthly_panel_max.dta / .csv

[MAX panel] rows=12,738, movies=193, months=66
  review_count not-null cells: 3,481
  treated (gvar_month notna) cells: 12,738

✨ All done.
